# COMBA Pipeline Benchmark
This notebook automates running the COMBA LangGraph pipeline multiple times to evaluate its average performance and stability.

**Objective:**
1. Run the pipeline 5 times on all modules using `txt` descriptions.
2. Collect cumulative metrics (Pass Rate, SC Trials, TS Trials, Iterations).
3. Calculate and display the averages across all runs.

In [ ]:
import subprocess
import os
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# --- Configuration ---
NUM_RUNS = 5
DESCRIPTION_TYPE = "txt"
MODULES_GLOB = "modules/*"
SUMMARY_FILE = f"reports/summary_langgraph.{DESCRIPTION_TYPE}.json"

print(f"Configured for {NUM_RUNS} runs using '{DESCRIPTION_TYPE}' descriptions.")

In [ ]:
all_runs_data = []

for i in range(1, NUM_RUNS + 1):
    print(f"\n[Run {i}/{NUM_RUNS}] Executing pipeline...")
    
    # Remove old summary to ensure we get fresh data
    if os.path.exists(SUMMARY_FILE):
        os.remove(SUMMARY_FILE)
    
    # Execute the pipeline script
    cmd = [
        "python", "run.py", "langgraph", 
        MODULES_GLOB, 
        "--descriptiontype", DESCRIPTION_TYPE
    ]
    
    # We use subprocess.run and capture output for debugging if needed
    result = subprocess.run(cmd, capture_output=False, text=True)
    
    if result.returncode != 0:
        print(f"Error in run {i}:")
        print(result.stderr)
        continue
    
    # Load the generated summary JSON
    if os.path.exists(SUMMARY_FILE):
        with open(SUMMARY_FILE, "r", encoding="utf-8") as f:
            summary_data = json.load(f)
            all_runs_data.append(summary_data)
    else:
        print(f"Warning: Summary file {SUMMARY_FILE} not found for run {i}.")

print("\nBenchmarking completed.")

## Data Aggregation and Average Calculation

In [ ]:
if not all_runs_data:
    print("No data collected to analyze.")
else:
    # Extract module names from the first run
    module_names = list(all_runs_data[0].keys())
    
    # Initialize cumulative structures
    # module_name -> { 'pass_count': 0, 'sc_total': 0, 'ts_total': 0, 'iter_total': 0 }
    module_stats = {name: {'pass_count': 0, 'sc_total': 0, 'ts_total': 0, 'iter_total': 0} for name in module_names}
    
    run_summaries = [] # Storing global stats per run
    
    for run_data in all_runs_data:
        run_pass_count = 0
        for name, details in run_data.items():
            # Samples property can be a list or a dict (if samples=1 in run.py)
            sample = details.get("samples")
            if isinstance(sample, list):
                sample = sample[0]
            
            if sample.get("final_status") == "pass":
                module_stats[name]['pass_count'] += 1
                run_pass_count += 1
            
            module_stats[name]['sc_total'] += sample.get("sc_trial", 0)
            module_stats[name]['ts_total'] += sample.get("ts_trial", 0)
            module_stats[name]['iter_total'] += sample.get("total_iter", 0)
        
        run_summaries.append(run_pass_count / len(module_names))

    # Build Final DataFrame
    summary_rows = []
    for name, stats in module_stats.items():
        summary_rows.append({
            "Module": name,
            "Avg Pass Rate (%)": (stats['pass_count'] / len(all_runs_data)) * 100,
            "Avg SC Trials": stats['sc_total'] / len(all_runs_data),
            "Avg TS Trials": stats['ts_total'] / len(all_runs_data),
            "Avg Total Iter": stats['iter_total'] / len(all_runs_data)
        })
    
    df = pd.DataFrame(summary_rows)
    
    # Global Metrics
    print("=== GLOBAL AVERAGES ===")
    print(f"Average Global Pass Rate: {np.mean(run_summaries)*100:.2f}%")
    print(f"Pass Rate Standard Deviation: {np.std(run_summaries)*100:.2f}%")
    print(f"Mean SC Trials (All Modules): {df['Avg SC Trials'].mean():.2f}")
    print(f"Mean TS Trials (All Modules): {df['Avg TS Trials'].mean():.2f}")
    
    display(df.sort_values("Avg Pass Rate (%)", ascending=False))